In [ ]:
# @title 1.1 🖥️ Check GPU & Configuration
import torch

# === CẤU HÌNH ===
LITE_MODE = False      # False = Full (2xT4), True = 4-bit only (1xT4)
LOAD_LLAVA = True      # LLaVA-Med
LOAD_DINO = True       # Grounding DINO
LOAD_SAM = True        # MedSAM
LOAD_BIOMED = True     # BiomedCLIP
# ================

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"✅ Detected {n_gpus} GPU(s):")
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        mem_gb = props.total_memory / 1024**3
        print(f"   GPU {i}: {props.name} ({mem_gb:.1f} GB)")
    
    if n_gpus < 2 and not LITE_MODE:
        print("⚠️ Cảnh báo: Chạy FULL mode trên 1 GPU có thể bị OOM.")
        print("   Khuyên dùng: GPU T4 x2 hoặc đặt LITE_MODE = True")
else:
    print("❌ Không tìm thấy GPU!")

In [ ]:
# @title 1.2 📦 Clone Repository & Install Dependencies
import os
import sys

# Clone repo
REPO_URL = "https://github.com/ngnam1104/TriMedAgent.git"

if not os.path.exists('TriMedAgent'):
    print("📥 Cloning TriMedAgent...")
    !git clone {REPO_URL}
else:
    print("✅ Repository exists.")

%cd TriMedAgent

# Install dependencies
print("\n📦 Installing dependencies...")
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q open_clip_torch einops timm safetensors sentencepiece pillow requests
!pip install -q gradio nest_asyncio protobuf scipy

if LOAD_DINO:
    print("   + Installing GroundingDINO...")
    !pip install -q groundingdino-py

if LOAD_SAM:
    print("   + Installing Segment Anything...")
    !pip install -q segment-anything

print("\n✅ Installation Complete.")
print("⚠️ Nếu có lỗi import, hãy Restart Session rồi chạy tiếp từ Cell 1.3")

In [ ]:
# @title 1.3 📥 Download Model Weights
import os
import requests
from tqdm import tqdm

def download_file(url, filepath):
    if os.path.exists(filepath):
        print(f"✅ Found {os.path.basename(filepath)}")
        return
    print(f"📥 Downloading {os.path.basename(filepath)}...")
    response = requests.get(url, stream=True)
    total = int(response.headers.get('content-length', 0))
    with open(filepath, 'wb') as f, tqdm(total=total, unit='iB', unit_scale=True) as bar:
        for chunk in response.iter_content(8192):
            f.write(chunk)
            bar.update(len(chunk))

os.makedirs("weights", exist_ok=True)

# Grounding DINO
if LOAD_DINO:
    download_file(
        "https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth",
        "weights/groundingdino_swint_ogc.pth"
    )
    download_file(
        "https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/groundingdino/config/GroundingDINO_SwinT_OGC.py",
        "weights/GroundingDINO_SwinT_OGC.py"
    )

# MedSAM
if LOAD_SAM:
    download_file(
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
        "weights/medsam_vit_b.pth"
    )

print("\n✅ All weights ready!")

In [ ]:
# @title 1.4 📚 Import from src/
import sys
from pathlib import Path

# Add project to path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import TriMedAgent modules
from src import (
    HybridReActOrchestrator,
    BiomedCLIPTool,
    GroundingDINOTool,
    MedSAMTool,
    LLaVATool,
    draw_boxes_on_image,
    draw_masks_on_image,
)
from src.utils import (
    KaggleConfig,
    get_device_map,
    print_gpu_info,
)

print("✅ Imports successful!")
print_gpu_info()

---
## 2️⃣ 🔧 Load Individual Tools

In [ ]:
# @title 2.4 🧠 Load LLaVA-Med (Visual Reasoning)
device_map = get_device_map()
llava_tool = None

if LOAD_LLAVA:
    print(f"🧠 Loading LLaVA-Med on {device_map['llava']}...")
    llava_tool = LLaVATool(
        device=device_map['llava'],
        quantize_4bit=True,  # 4-bit quantization for T4
        load_on_init=True
    )
    print("✅ LLaVA-Med ready!")
else:
    print("⚠️ LLaVA disabled")

In [ ]:
# @title 2.1 🔬 Load BiomedCLIP (Triage)
import torch


biomedclip_tool = None

if LOAD_BIOMED:
    print(f"🔬 Loading BiomedCLIP on {device_map['biomedclip']}...")
    biomedclip_tool = BiomedCLIPTool(
        device=device_map['biomedclip'],
        load_on_init=True
    )
    print("✅ BiomedCLIP ready!")
else:
    print("⚠️ BiomedCLIP disabled")

In [ ]:
# @title 2.2 🎯 Load Grounding DINO (Detection)

dino_tool = None

if LOAD_DINO:
    print(f"🎯 Loading Grounding DINO on {device_map['grounding_dino']}...")
    dino_tool = GroundingDINOTool(
        device=device_map['grounding_dino'],
        config_path="weights/GroundingDINO_SwinT_OGC.py",
        checkpoint_path="weights/groundingdino_swint_ogc.pth",
        box_threshold=0.25,
        load_on_init=True
    )
    print("✅ Grounding DINO ready!")
else:
    print("⚠️ Grounding DINO disabled")

In [ ]:
# @title 2.3 🎭 Load MedSAM (Segmentation)

medsam_tool = None

if LOAD_SAM:
    print(f"🎭 Loading MedSAM on {device_map['medsam']}...")
    medsam_tool = MedSAMTool(
        device=device_map['medsam'],
        checkpoint_path="weights/medsam_vit_b.pth",
        model_type="vit_b",
        load_on_init=True
    )
    print("✅ MedSAM ready!")
else:
    print("⚠️ MedSAM disabled")

---
## 3️⃣ 🧪 Test Individual Tools

In [ ]:
# @title 3.1 📷 Load Sample Image
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

SAMPLE = "/kaggle/working/TriMedAgent/images/example_chest.jpg"
sample_image = Image.open(SAMPLE).convert("RGB")
print(f"✅ Image size: {sample_image.size}")

plt.figure(figsize=(6, 6))
plt.imshow(sample_image)
plt.title("Sample Medical Image")
plt.axis('off')
plt.show()

In [ ]:
# @title 3.2 🔬 Test BiomedCLIP (Triage)

if biomedclip_tool:
    print("🔬 Running Triage...")
    triage_result = biomedclip_tool.triage(sample_image)
    
    print(f"\n📊 Triage Result:")
    print(f"   Modality: {triage_result['modality']}")
    print(f"   Confidence: {triage_result['modality_confidence']:.1%}")
    
    print(f"\n   Top 5 Modality Scores:")
    for label, score in list(triage_result['modality_scores'].items())[:5]:
        bar = "█" * int(score * 30)
        print(f"   {label}: {score:.1%} {bar}")
        
    if 'abnormality' in triage_result:
        print(f"\n   Abnormality: {triage_result['abnormality']}")
        print(f"   Confidence: {triage_result['abnormality_confidence']:.1%}")
else:
    print("⚠️ BiomedCLIP not loaded")

In [ ]:
# @title 3.3 🎯 Test Grounding DINO (Detection)
import matplotlib.patches as patches
import numpy as np

if dino_tool:
    print("🎯 Running Detection...")
    query = "lung nodule"
    
    result = dino_tool.detect(sample_image, query, return_phrases=True)
    
    print(f"\n📊 Detection Result:")
    print(f"   Query: '{query}'")
    print(f"   Boxes found: {len(result.get('boxes', []))}")
    
    # Visualize
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(sample_image)
    
    boxes = result.get('boxes', [])
    scores = result.get('scores', [])
    
    for i, (box, score) in enumerate(zip(boxes[:5], scores[:5])):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor='red', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1-5, f"{score:.2f}", color='white', fontsize=10,
                bbox=dict(facecolor='red', alpha=0.7))
    
    ax.set_title(f"Detection: '{query}'")
    ax.axis('off')
    plt.show()
else:
    print("⚠️ Grounding DINO not loaded")

In [ ]:
# @title 3.4 🎭 Test MedSAM (Segmentation)

if medsam_tool and 'result' in dir() and result.get('boxes'):
    print("🎭 Running Segmentation...")
    
    test_boxes = result['boxes'][:3]  # Top 3 boxes
    masks = medsam_tool.segment(sample_image, boxes=test_boxes)
    
    print(f"   Generated {len(masks)} masks")
    
    # Visualize
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(sample_image)
    
    overlay = np.zeros((*np.array(sample_image).shape[:2], 4))
    for mask in masks:
        overlay[mask > 0] = [1, 0, 0, 0.4]
    ax.imshow(overlay)
    
    for box in test_boxes:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor='yellow', facecolor='none', linestyle='--'
        )
        ax.add_patch(rect)
    
    ax.set_title(f"MedSAM Segmentation ({len(masks)} masks)")
    ax.axis('off')
    plt.show()
else:
    print("⚠️ Run detection first or MedSAM not loaded")

In [ ]:
# @title 3.5 🧠 Test LLaVA-Med (Visual QA)

if llava_tool:
    print("🧠 Asking LLaVA...")
    
    question = "Describe any abnormalities you see in this chest X-ray image."
    response = llava_tool.query(sample_image, question)
    
    print(f"\n📊 LLaVA Response:")
    print(f"   Q: {question}")
    print(f"   A: {response}")
else:
    print("⚠️ LLaVA not loaded")

---
## 4️⃣ 🎯 Hybrid ReAct Orchestrator

In [ ]:
# @title 4.1 🚀 Initialize Orchestrator

device_map = get_device_map()

orchestrator = HybridReActOrchestrator(
    llava_device=device_map['llava'],
    tool_device=device_map['grounding_dino'],
    max_iterations=3,
    enable_verification=True,
    enable_rag=False,  # RAG needs API key
)

print("🎯 Orchestrator initialized")
print(orchestrator)

In [ ]:
# @title 4.2 📦 Load All Tools into Orchestrator

print("📦 Loading tools into orchestrator...")
orchestrator.load_tools(
    load_biomedclip=LOAD_BIOMED,
    load_dino=LOAD_DINO,
    load_medsam=LOAD_SAM,
    load_llava=LOAD_LLAVA,
    load_rag=False,
    llava_quantize=True,
)

print("\n✅ Orchestrator ready!")
print(orchestrator)

In [ ]:
# @title 4.3 🔬 Run Full Pipeline
import time

print("🔬 Running Hybrid ReAct Pipeline...")
print("="*50)

query = "Find any nodules or suspicious lesions in the lungs"

start_time = time.time()
result = orchestrator.process(sample_image, query)
elapsed = time.time() - start_time

print(f"\n📊 Pipeline Result:")
print(f"   Success: {result.success}")
print(f"   Time: {elapsed:.1f}s")
print(f"   Steps: {' → '.join(result.steps_executed)}")
print(f"   Iterations: {result.agent_iterations}")
print(f"   Modality: {result.triage_modality} ({result.triage_confidence:.0%})")
print(f"   Boxes found: {len(result.verified_boxes)}")
print(f"   Masks generated: {len(result.masks)}")

if result.strategic_plan:
    plan = result.strategic_plan
    print(f"\n📋 Strategic Plan:")
    print(f"   Target: {plan.target_object}")
    print(f"   Size: {plan.target_size.value}")
    print(f"   Strategy: {plan.strategy.value}")
    print(f"   Location: {plan.anatomical_location or 'N/A'}")

In [ ]:
# @title 4.4 📊 Visualize Results

if result.annotated_image:
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    axes[0].imshow(sample_image)
    axes[0].set_title("Original")
    axes[0].axis('off')
    
    axes[1].imshow(result.annotated_image)
    axes[1].set_title(f"Detected: {len(result.verified_boxes)} regions")
    axes[1].axis('off')
    
    plt.suptitle(f"Query: '{query}'")
    plt.tight_layout()
    plt.show()
else:
    print("No annotated image generated")

In [ ]:
# @title 4.5 📝 View Analysis Report

if result.llava_analysis:
    print("🔬 LLaVA Analysis:")
    print("-" * 50)
    print(result.llava_analysis)

if result.final_report:
    print("\n📋 Final Report:")
    print("-" * 50)
    print(result.final_report)

---
## 5️⃣ 💬 Interactive Demo (Optional)

In [ ]:
# @title 5.1 🎨 Interactive Query Function

def analyze_image(image_path_or_url, query):
    """Analyze medical image with custom query."""
    from PIL import Image
    import requests
    from io import BytesIO
    
    # Load image
    if image_path_or_url.startswith("http"):
        response = requests.get(image_path_or_url)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_path_or_url).convert("RGB")
    
    # Process
    result = orchestrator.process(image, query)
    
    # Display
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(image)
    axes[0].set_title("Input")
    axes[0].axis('off')
    
    if result.annotated_image:
        axes[1].imshow(result.annotated_image)
    else:
        axes[1].imshow(image)
    axes[1].set_title(f"Result ({len(result.verified_boxes)} detections)")
    axes[1].axis('off')
    
    plt.suptitle(f"Query: {query}")
    plt.tight_layout()
    plt.show()
    
    if result.final_report:
        print("\n📋 Report:")
        print(result.final_report)
    
    return result

print("✅ Function ready!")
print("Usage: analyze_image('image_url_or_path', 'your query')")

In [ ]:
# @title 5.2 🧪 Test with Custom Query

# Uncomment and modify to test:
# result = analyze_image(
#     "https://your-image-url.com/image.jpg",
#     "Find any fractures in the bone"
# )

---
## 🎨 Gradio Web Interface

Launch beautiful web UI for TriMedAgent

In [ ]:
!cd /kaggle/working/TriMedAgent && git pull

In [ ]:
# @title 🚀 Launch Gradio Web Interface (share=True for Kaggle public URL)

from src import launch_demo

# Launch with public URL for Kaggle environment
# Set share=True to get a public URL that works through Kaggle's firewall
demo = launch_demo(
    orchestrator, 
    share=True,  # Creates public ngrok URL
    port=7860
)

# The URL will be printed in the output
# Example: "Running on public URL: https://xxxxx.gradio.live"

In [ ]:
# @title 🛑 Stop Gradio Server (Optional)

# Uncomment to stop the Gradio server:
# demo.close()
# print("✅ Gradio server stopped")

---
## 6️⃣ 🧹 Cleanup

In [ ]:
# @title 6.1 🗑️ Unload Models (Free GPU Memory)

# Uncomment to free memory:
# orchestrator.unload_all()
# print("✅ All models unloaded")